<a href="https://colab.research.google.com/github/parsa-asgari/bahar-bootcamp/blob/main/bahar_RAG_LLM_plain_docx.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Practice: RAG + LLM test

## 1. Install python packages

In [1]:
!pip install pinecone openai python-docx
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 101.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pinecone]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 27.8 MB/s  0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.20.2
    Uninstalling transformers-5.20.2:
      Successfully uninstalled transformers-5.20.2


## 2. API Keys

In [2]:
# API KEYS
PINECONE_API_KEY=''
PINECONE_INDEX_NAME=''
OPENROUTER_API_KEY=''
GROQ_API_KEY=''
HUGGINGFACE_TOKEN=''

## 3. read bahar_danesh_internal_policy.docx word file

In [22]:
from docx import Document


def read_docx(file_path: str) -> str:
    doc = Document(file_path)
    content = []

    # Extract paragraphs
    for paragraph in doc.paragraphs:
        text = paragraph.text.strip()
        if text:
            content.append(text)

    # Extract tables
    for table in doc.tables:
        for row in table.rows:
            cells = [
                cell.text.strip().replace("\n", " ")
                for cell in row.cells
            ]
            content.append(" | ".join(cells))

    return "\n\n".join(content)


if __name__ == "__main__":
    text = read_docx("bahar_danesh_internal_policy.docx")
    print(text[:5])


شرکت 


In [47]:
from sentence_transformers import SentenceTransformer
sentences = 'مدل های زبانی بزرگ چه هستند؟'

model = SentenceTransformer('heydariAI/persian-embeddings')
sentence_emb = model.encode(sentences)
print(sentence_emb)
print(type(sentence_emb))
print(sentence_emb.shape)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[ 0.07500484 -0.20403147  1.2002752  ...  0.32945707  1.4784025
  0.16393864]
<class 'numpy.ndarray'>
(1024,)


## 4. Insert to Pinecone

In [23]:
import os, time
from pinecone import Pinecone, SchemaBuilder

pc = Pinecone(api_key=PINECONE_API_KEY)

schema = (
    SchemaBuilder()
      .add_dense_vector_field(name="embedding", dimension=1024, metric="cosine")
      .build()
)
if not pc.indexes.exists(name=PINECONE_INDEX_NAME):
    pc.indexes.create(name=PINECONE_INDEX_NAME, schema=schema)

while not pc.indexes.describe(name=PINECONE_INDEX_NAME).status.ready:
    time.sleep(2)

index = pc.Index(name=PINECONE_INDEX_NAME)

In [49]:
for line in text.split('\n')[:5]:
  print(line)

شرکت بهار دانش

معاونت اداری و سرمایه انسانی

سند شماره: BK-HR-1405-017


In [52]:
import os
from pathlib import Path
from pinecone import Pinecone

# Pinecone client
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(PINECONE_INDEX_NAME)
chunks_inserted = 0
for i,line in enumerate(text.split('\n')):
    if len(line):
      embedding = model.encode(line)
      document = [{
              "_id": f"{i}",
              "embedding": embedding.tolist(),
              "text": line,
              "source": "bahar_danesh_internal_policy.docx",
          }]
      result = index.documents.upsert(
          namespace="__default__",
          documents=document,
      )
      chunks_inserted = chunks_inserted + 1

print(
    f"Uploaded {chunks_inserted} chunks"
)
print("Finished uploading documents.")


Uploaded 89 chunks
Finished uploading documents.


## 5. Read from Pinecone

In [60]:
import os
from pathlib import Path
from pinecone import Pinecone
from pinecone import DenseVectorQuery

# Pinecone client
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(PINECONE_INDEX_NAME)

query = 'ساعات کاری'
model = SentenceTransformer('heydariAI/persian-embeddings')
sentence_emb = model.encode(query)
results = index.documents.search(
    namespace="__default__",
    top_k=10,
    score_by=[DenseVectorQuery(field="embedding", values=sentence_emb)],
    include_fields=["text", "source"],
)
results[:5]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Matches:,10
Namespace:,__default__
Read Units:,1


In [84]:
for doc in results.matches:
  print(doc.text)

۳. ساعات کاری و حضور
۳-۱. ساعات کاری عادی از شنبه تا چهارشنبه، ساعت ۸:۳۰ تا ۱۷:۰۰ است.
۳-۲. زمان استراحت روزانه از ساعت ۱۲:۳۰ تا ۱۳:۱۵ تعیین شده و جزو زمان کاری محسوب نمی‌شود.
موضوع: تشریح ساعات کاری، مرخصی، دورکاری، آموزش، مزایا و فرایندهای اداری کارکنان
پرسش: حداکثر چند روز در هفته می‌توانم دورکار باشم؟ پاسخ: حداکثر دو روز، با موافقت مدیر مستقیم و با رعایت نیازهای واحد.
۵-۲. روزهای دورکاری باید با موافقت مدیر مستقیم تعیین شوند و در تقویم تیم ثبت شوند.
۵. دورکاری
۴-۴. مرخصی ساعتی برای بازه‌های حداقل یک ساعت قابل ثبت است. درخواست‌های کمتر از یک ساعت در این سناریو به‌عنوان خروج موقت ثبت می‌شوند و باید به تأیید مدیر برسند.
پرسش: مدیر تا چه زمانی باید به درخواست مرخصی پاسخ دهد؟ پاسخ: حداکثر دو روز کاری پس از ثبت درخواست.
• آیا کارمند پشتیبانی مشتریان بدون توجه به شیفت می‌تواند دو روز دورکاری داشته باشد؟


## 6. Interface with LLM

In [78]:
from openai import OpenAI
# LLM
llm_client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
)

MODEL = "openai/gpt-oss-20b"

response = llm_client.responses.create(
    model=MODEL,
    instructions="you are a helpful assistant.",
    input="howdy! how was your day?",
)


In [77]:
print(response.output[1].content[0].text)

Howdy! Thanks for asking—my day’s been going pretty smoothly. I’ve been helping folks with everything from recipe ideas to tech questions, so it’s been a nice mix of learning and assisting. How about you? How’s your day been going?


In [87]:
## Doing a real RAG

PROMPT = '''
*INSTRUCTION*
You are a helpful assistant representing 'بهار دانش', a telecom company in tehran, iran.
You are deployed to answer user questions.
Greet the user, if you have no conversation history.
Be polite, be gentle.
use the context to answer user questions as best as you can.
Don't answer questions unrelated to 'بهار دانش'.

*CONTEXT*
{context}
'''

query = "در مورد ساعات کاری شرکت به من بگو"

model = SentenceTransformer('heydariAI/persian-embeddings')
sentence_emb = model.encode(query)
results = index.documents.search(
    namespace="__default__",
    top_k=10,
    score_by=[DenseVectorQuery(field="embedding", values=sentence_emb)],
    include_fields=["text", "source"],
)

context = [doc.text for doc in results.matches]


response = llm_client.responses.create(
    model=MODEL,
    instructions=PROMPT.format(context=context),
    input="در مورد ساعات کاری شرکت به من بگو",
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [88]:
print(response.output[1].content[0].text)

سلام!  
ساعات کاری معمول شرکت به شرح زیر است:

- **روزهای کاری**: شنبه تا چهارشنبه  
- **ساعت شروع و پایان**: ۸:۳۰ صبح تا ۱۷:۰۰ عصر  
- **زمان استراحت روزانه**: از ساعت ۱۲:۳۰ تا ۱۳:۱۵ که جزو ساعت کاری محسوب نمی‌شود.  

اگر سوال دیگری در مورد ساعات یا مقررات کار دارید، خوشحال می‌شوم کمک کنم!
